# 00 Setup and verification

**Status: NOT RUN ON COLAB.** Generated by `scripts/make_notebooks.py`. Any output cells in the repository copy come from a local CPU smoke run (`SMOKE=True`, 20 samples), never from a real run.

| | |
|---|---|
| Purpose | Mount Drive, check every input file, record the environment and GPU. |
| Inputs | `MyDrive/KDU/Thesis/` (Dataset, run folders) |
| Outputs | `showcase_runs/00_setup_and_verify/env_colab.json`, `input_hashes.json` |
| Runtime (T4 GPU, estimate) | 1 min |
| Fills in `thesis_numbers.json` | T7.env_colab |

Run the cells top to bottom. Set `SMOKE = True` in the setup cell for a dry run on 20 samples. Every run writes a manifest with input and output hashes; the last cell prints an evidence table read from the saved files.

In [ ]:
NOTEBOOK = "00_setup_and_verify"
NEEDS_GPU = False      # True: the notebook stops at once if Colab gave no GPU

# ---- setup (same in every notebook) ----
import os, sys, json, ast, time, hashlib, random, platform, subprocess, re
from pathlib import Path
import numpy as np, pandas as pd

SMOKE = os.environ.get("TTPMAP_SMOKE", "0") == "1"      # set SMOKE = True by hand for a 20-sample dry run
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("Drive mount failed:", type(e).__name__, e)
    if not os.path.isdir("/content/drive/MyDrive") and not SMOKE:
        raise RuntimeError("Google Drive is not mounted. The Colab extension for VS Code cannot mount Drive (known limitation of the extension), "
                           "and these notebooks read their inputs from and save their results to Drive. Open this notebook in the browser instead: "
                           "https://colab.research.google.com > File > Upload notebook (or open it from Drive). See notebooks/COLAB_RUNBOOK.md, section 0.")
BASE = Path("/content/drive/MyDrive/KDU/Thesis") if IN_COLAB else Path(os.environ.get("TTPMAP_ROOT", r"E:\Osamas Paper"))
DATA_DIR, CLS_ROOT, MATCH_ROOT, SHAP_DIR = BASE / "Dataset", BASE / "Exp_1_4_run_tram_2_epoc_30", BASE / "run_match_1", BASE / "Exp_3_0_shap"
MATCH2 = BASE / "run_match_2_saved"                      # NEW folder: matcher retrained WITH saved weights (notebook 04)
OUT = (Path(os.environ["TTPMAP_SMOKE_OUT"]) if os.environ.get("TTPMAP_SMOKE_OUT") else BASE / "showcase_runs") / NOTEBOOK
OUT.mkdir(parents=True, exist_ok=True)
SEEDS = (42, 43, 44)
if IN_COLAB and not SMOKE and not (BASE / "Dataset" / "tram_test.tsv").exists():
    _msg = (f"Project files not found under {BASE}. Upload the folder E:\\Osamas Paper\\_showcase\\drive_upload\\KDU into 'My Drive' "
            "(drive.google.com, same Google account as this Colab session), then run this cell again. See notebooks/COLAB_RUNBOOK.md, section 0.")
    if NOTEBOOK == "00_setup_and_verify": print("WARNING:", _msg)
    else: raise RuntimeError(_msg)

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
    except ImportError:
        pass

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for b in iter(lambda: fh.read(1 << 20), b""): h.update(b)
    return h.hexdigest()

def env_info():
    info = {"python": platform.python_version(), "platform": platform.platform(), "in_colab": IN_COLAB, "smoke": SMOKE}
    try:
        import torch
        info.update(torch=torch.__version__, cuda=torch.cuda.is_available(), gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
    except ImportError:
        pass
    return info

def write_manifest(run_id, system, seed, config, inputs, outputs, metrics, status):
    """One manifest per run: what went in, what came out, with hashes. Read later by the ingest step."""
    m = {"run_id": run_id, "system": system, "seed": seed, "config_hash": hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()[:16], "config": config,
         "env": {k: str(v) for k, v in env_info().items()}, "input_sha256": {str(p): sha256(p) for p in inputs if Path(p).exists()},
         "output_sha256": {str(p): sha256(p) for p in outputs if Path(p).exists()}, "metrics": {k: float(v) for k, v in metrics.items()},
         "status": "SMOKE" if SMOKE else status, "written": time.strftime("%Y-%m-%dT%H:%M:%S")}
    (OUT / f"manifest_{run_id}.json").write_text(json.dumps(m, indent=1)); return m

print(f"notebook={NOTEBOOK} | SMOKE={SMOKE} | IN_COLAB={IN_COLAB} | BASE={BASE} | OUT={OUT}")
print(env_info())
import importlib.metadata as _md
print("packages:", {p: (_md.version(p) if p in {d.metadata["Name"].lower() for d in _md.distributions() if d.metadata["Name"]} else "not installed")
                    for p in ("torch", "transformers", "scikit-learn", "numpy", "pandas", "shap", "scipy")})
_gpu = env_info().get("cuda", False)
if NEEDS_GPU and not _gpu and not SMOKE:
    raise RuntimeError("No GPU found. In Colab: Runtime > Change runtime type > T4 GPU > Save, then run this cell again. (Set SMOKE = True only for a dry run.)")
if not NEEDS_GPU and not _gpu:
    print("note: no GPU; this notebook runs on CPU in a few minutes")

In [ ]:
need = [DATA_DIR / "tram_dev.tsv", DATA_DIR / "tram_test.tsv", DATA_DIR / "ttp-desc-mappings.csv", DATA_DIR / "combined_train.csv", SHAP_DIR / "shap_v2_roberta_pw_s42.jsonl",
        MATCH_ROOT / "zeroshot" / "test_scores.npy"] + [CLS_ROOT / f"{t}_pw_s{s}" / f for t in ("roberta", "secbert") for s in SEEDS for f in ("test_probs.npy", "dev_probs.npy", "metrics.json", "mlb_classes.npy")] \
       + [MATCH_ROOT / f"ft20_s{s}" / f"{sp}_scores.npy" for s in SEEDS for sp in ("dev", "test")]
optional = [DATA_DIR / "tram_train.tsv"] + [CLS_ROOT / f"{t}_pw_s42" / "model.safetensors" for t in ("roberta", "secbert")] + [MATCH2 / "ft20_s42" / "matcher_state.pt"]
missing = [str(p) for p in need if not p.exists()]
print("required files:", len(need), "| missing:", len(missing)); [print("  MISSING", m) for m in missing]
for p in optional: print(("  present " if p.exists() else "  absent  ") + str(p))
hashes = {str(p.relative_to(BASE)): sha256(p) for p in need if p.exists() and p.stat().st_size < 50e6}
(OUT / "input_hashes.json").write_text(json.dumps(hashes, indent=1))

In [ ]:
pkgs = {}
try:
    import importlib.metadata as md
    pkgs = {d.metadata["Name"]: d.version for d in md.distributions() if d.metadata["Name"]}
except Exception as e:
    print("package listing failed:", e)
gpu = ""
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], capture_output=True, text=True, timeout=30).stdout.strip()
except Exception:
    gpu = "nvidia-smi not available"
env = {**env_info(), "nvidia_smi": gpu, "n_packages": len(pkgs), "packages": dict(sorted(pkgs.items())), "missing_required_files": missing}
(OUT / ("env_colab.json" if IN_COLAB else "env_local.json")).write_text(json.dumps(env, indent=1))
print({k: v for k, v in env.items() if k != "packages"})

In [ ]:
# ---- evidence table: read back from the files this notebook wrote (no typed numbers) ----
rows = []
for p in sorted(OUT.glob("*.json")):
    d = json.loads(p.read_text())
    flat = d.get("metrics", d) if isinstance(d, dict) else {}
    for k, v in flat.items():
        if isinstance(v, (int, float)) and not isinstance(v, bool): rows.append({"file": p.name, "key": k, "value": round(float(v), 4)})
ev = pd.DataFrame(rows)
print(f"status: {'SMOKE (not a result)' if SMOKE else 'RUN'} | files in {OUT}: {len(list(OUT.glob('*')))}")
print(ev.to_string(index=False) if len(ev) else "no numeric outputs")